このページではWord2Vecとロジスティック回帰を使って英文フェイクニュースの識別を行う。<br>
使用する仮想環境のバージョンは**python 3.11**です。<br>
この機械学習で使うWord2Vecについて
# Word2Vec


作成した変数を取り出す

In [3]:
import pickle

with open("../../Data/train_test_split.pkl", "rb") as f:
    x_train, x_test, y_train, y_test = pickle.load(f)

テキストデータの前処理をそれぞれ実行<br>
nltkをコマンドでインストール -> pip install nltk<br>
nltk.download("stopwords")を一回実行

In [4]:
import re
from nltk.corpus import stopwords


def NormalizeText(text):
    text = text.lower() #テキストを小文字変換
    text = text.replace('\n', ' ') #改行をスペースに置換
    text = re.sub(r'[^\w\s]', '', text) #特殊文字を削除
    text = text.strip() #テキストの前後の空白を削除
    return text

def TokenizeText(text):
    tokens = re.findall(r'\b\w+\b',text) #テキストをスペースで分割してトークン化
    return tokens

def StopWordRemoval(tokens):
    stop_words = set(stopwords.words('english')) #英語のストップワードを取得
    filtered_tokens = [word for word in tokens if word not in stop_words] #ストップワードを除去
    return filtered_tokens

def Preprocessing(x_text): #前処理をまとめたもの
    x_text_Normalized = x_text.apply(NormalizeText) #訓練用テキストを正規化
    x_text_Tokenized = x_text_Normalized.apply(TokenizeText) #訓練用テキストをトークン化
    x_text_Filtered = x_text_Tokenized.apply(StopWordRemoval) #ストップワードを除去
    return x_text_Filtered

x_train_Filtered = Preprocessing(x_train) #前処理を実行
x_test_Filtered = Preprocessing(x_test)


文脈に基づいて単語間の関係を学習し、各単語をベクトル化したモデルを作成する

In [11]:
from gensim.models import Word2Vec

def Size_Word2Vec(x_train_Filtered, size):
    model = Word2Vec(
        x_train_Filtered,   #データ
        vector_size=size,   #ベクトル次元
        window=5,           #文章ウィンドウ
        min_count=1,        #出現頻度の閾値
        sg=1                #1=skip-gram, 0=CBOW
    )
    return model

model = Size_Word2Vec(x_train_Filtered, 300)


単語ベクトルを返すWord2Vecを使ってベクトルの平均として文章ベクトルをつくる

In [12]:
import numpy as np
from sklearn.linear_model import LogisticRegression

def document_vector(model, doc): #文章をベクトルに変換する関数
    #モデルに存在する単語だけの平均
    vectors = [model.wv[word] for word in doc if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

#文章ベクトルの集合をそれぞれ求める
x_train_vec = [document_vector(model, doc) for doc in x_train_Filtered]
x_test_vec = [document_vector(model, doc) for doc in x_test_Filtered]

文章ベクトルの集合をつかってロジスティック回帰で学習・分類

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

def Learning(x_vec, y_mapped):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_vec, y_mapped)
    return clf

def DispResult(clf, x_test_vec, y_test_mapped):
    y_pred = clf.predict(x_test_vec)
    print(classification_report(y_test_mapped, y_pred))
    accuracy = accuracy_score(y_test_mapped, y_pred)
    return accuracy


clf = Learning(x_train_vec, y_train)
DispResult(clf, x_test_vec, y_test)

              precision    recall  f1-score   support

           0       0.99      0.96      0.97      3472
           1       0.97      0.99      0.98      4258

    accuracy                           0.98      7730
   macro avg       0.98      0.98      0.98      7730
weighted avg       0.98      0.98      0.98      7730



0.9772315653298835